<a href="https://colab.research.google.com/github/joyashre/LLM-NLP-Learning-task/blob/main/NLP_Task_2_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# NLP Task 2

**QUESTION:** Use pretrained Causal LM models (like llama, gemma, etc) from huggingface to implement

1. Sentiment Analysis
2. Text Summarization
3. Question Answering

In [ ]:
pip install transformers torch accelerate

In [ ]:
import textwrap
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

In [ ]:
# CONFIG — change model here
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"  # GPU is available or not checking it otherwise use CPU ,  GPU is faster than CPU for AI LLM model
DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32    #trying to use float 16 so that memory will not full.. otherwise it will give OUT OF MEMORY error.

print(f"Loading model : {model_name}")
print(f"Device        : {DEVICE}")
print(f"Dtype         : {DTYPE}\n")

tokenizer = AutoTokenizer.from_pretrained(model_name) # token krne k liye  text to number because Neural network does not understand language , it understand numbers
model = AutoModelForCausalLM.from_pretrained(           # predict the next word ... casually do it
    model_name,
    torch_dtype=DTYPE,
    device_map="auto",      # automatically maps layers to available devices  matlab konsa hissa GPU par jayega konsa RAM per
)
model.eval()   # for stop random behaviour like dropout.. for safe the memory
print("Model loaded successfully.\n")  # just give peace of our mind .. to see that above heavy code are successfully run.

Loading model : TinyLlama/TinyLlama-1.1B-Chat-v1.0
Device        : cuda
Dtype         : torch.float16



/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Model loaded successfully.



In [ ]:
print(tokenizer)

TokenizersBackend(name_or_path='TinyLlama/TinyLlama-1.1B-Chat-v1.0', vocab_size=32000, model_max_length=2048, padding_side='right', truncation_side='right', special_tokens={'bos_token': '<s>', 'eos_token': '</s>', 'unk_token': '<unk>', 'pad_token': '</s>'}, added_tokens_decoder={
	0: AddedToken("<unk>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	1: AddedToken("<s>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	2: AddedToken("</s>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}
)


why we do print (tokenizer)

hume pata hona chahiye ki uski limits kya hain, uski dictionary kitni badi hai, aur wo kitne words ek baar mein samajh sakta hai. Ye print karke hum basically uski rulesbook padh rahe hain.

what I found after print (tokenizer)

1. vocab_size=32000 (Model ki Dictionary)

Matlab: Is TinyLlama model ko total 32,000 alag-alag tokens (words ya words ke tukde) aate hain.

Agar aap koi aisi spelling likhti hain jo in 32,000 mein nahi hai, toh tokenizer usko tod kar chote hisson mein baant dega jo uski dictionary mein hain.

2. model_max_length=2048 (Model ki Memory/Limit)

Matlab: Ye sabse important setting hai! Iska matlab hai ki ye model ek baar mein maximum 2048 tokens (lagbhag 1500 English words) ko hi process kar sakta hai.

Agar aap isko 2048 tokens se bada PDF ya lamba text dengi, toh ya toh code error dega, ya aage ka text kaat dega.

3. padding_side='right', truncation_side='right' (Formatting Rules)

AI models hamesha fixed size mein data lete hain.

Padding (right): Agar aapka sawal chota hai (jaise sirf 10 tokens ka), toh model baaki bachi hui jagah ko bharne ke liye right side (end mein) blank spaces (pad tokens) laga dega.

Truncation (right): Agar aapka sawal 2048 limit se bada ho gaya, toh model right side (end se) words ko kaat (truncate) dega.

4. special_tokens aur added_tokens_decoder (Secret Codes)

AI models ko batana padta hai ki kab bolna shuru karna hai aur kab rukna hai. Inke liye kuch special tags hote hain:

bos_token: $"<s>"$ (Begin Of Sentence): Jab AI apna jawab dena shuru karega, toh wo internal system mein sabse pehle $<s>$ lagayega. Iska ID Number 1 hai.

eos_token: $'</s>'$ (End Of Sentence): Jab AI ka jawab khatam ho jayega, toh wo $</s>$ generate karega. Ye dekh kar system samajh jayega ki ab aur words generate nahi karne hain aur output rok dega. Iska ID Number 2 hai.

unk_token: $'<unk>'$ (Unknown): Agar aapne aisi bhasha (jaise alien language) likh di jo usko bilkul samajh nahi aayi, toh wo usko $<unk>$ bana dega.

In [ ]:
def generate(prompt: str, max_new_tokens: int = 256, temperature: float = 0.3) -> str:
    # prompt: str → input text (user question)
    # max_new_tokens → maximum number of words/tokens to generate ... itna lamba jawab chahiye. 256 tokens ka matlab hai lagbhag 150-200 words.
    # temperature: float = 0.3 → controls randomness 0.0= strict , 1 = creative
    # -> str → returns generated text

    #Checking whether the tokenizer supports a chat format
    if hasattr(tokenizer, "apply_chat_template") and tokenizer.chat_template:
        # if yes -> Converts prompt into a chat-style format
        messages = [{"role": "user", "content": prompt}]  #ye meri sawal ko aise format kar dega: <|user|>\nWhat is ML?<|assistant|>\n. Isse model ko samajh aata hai ki "Achha, user ne sawal pucha hai, ab mujhe assistant ban kar jawab dena hai.""
        formatted = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
    else:
        # if no -> Uses plain text as input
        formatted = prompt

    # Converts text into tokens (numbers) => return_tensors="pt" → PyTorch tensors => .to(DEVICE) → moves to CPU/GPU
    inputs = tokenizer(formatted, return_tensors="pt").to(DEVICE)
    # to separate new output from input.... because ab ye jawab dete hain, toh output mein Humara Sawal + AI ka Jawab dono ek saath chipke hue aate hain.
    input_len = inputs["input_ids"].shape[1]

    # Disabling gradient calculation → faster & less memory
    with torch.no_grad():
        output_ids = model.generate(   #predict next word one by one and generate
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=temperature > 0,
            pad_token_id=tokenizer.eos_token_id, # eos = end of sentence
        )

    new_ids = output_ids[0][input_len:] # input katke naya output le rh h
    return tokenizer.decode(new_ids, skip_special_tokens=True).strip() #convert in text again
    # skip_special_tokens=True: Ye <s>, </s> jaise system tags ko hata deta hai taaki hume ekdum saaf aur natural text mile.
    # strip() aage-peeche ki faltu spaces hata deta hai.

1. **inputs (The Python Unpacker)

Python mein ** ko "Dictionary Unpacking" kehte hain.

Jab aapne upar inputs = tokenizer(...) kiya tha, toh tokenizer ne ek Python Dictionary (dict) banayi thi jisme do main cheezein thi:

input_ids (aapke words ke numbers)

attention_mask (0s aur 1s jo model ko batate hain ki kin words par dhyan dena hai).

Agar aap sirf inputs likhti, toh function ko lagta aap ek single dictionary object bhej rahi hain aur wo error de deta.

**inputs likhne se Python automatically us dictionary ko khol deta hai aur model ko aise bhejta hai: input_ids=..., attention_mask=.... Ye code ko chota aur clean rakhne ka ek smart tareeqa hai.

2. do_sample=temperature > 0,

Yahan do raaste hote hain:

Greedy Search (do_sample=False): AI hamesha sabse high probability wala word (Apple) hi chunega. Ye boring hota hai, robtic lagta hai, par maths aur facts ke liye best hai.

Sampling (do_sample=True): AI thoda dice roll karega. Wo zyada tar "Apple" chunega, par kabhi kabhi "Banana" ya "Orange" bhi chun lega. Isse output natural aur creative lagta hai.

humne temperature 0.3 choose kiya matlab true , matlab thoda creative.

3. 3. pad_token_id=tokenizer.eos_token_id (The Warning Silencer)

LLMs ko maths karne ke liye fixed matrix size chahiye hota hai. Agar sentence chota hai, toh usko bachi hui khali jagah (empty spaces) mein "pad tokens" (filler words) bharne padte hain.

Problem: TinyLlama, Llama, aur Mistral jaise naye models ki dictionary mein by default koi alag se "pad_token" hota hi nahi hai.

Agar aap ye line nahi likhengi, toh model apna kaam toh karega, lekin PyTorch aapki screen par lambi-lambi lal rang ki warnings dena shuru kar dega ki "pad_token is missing".

Solution: Hum model ko bolte hain ki "Bhai, jab bhi tujhe array mein khali jagah bharni ho, toh tu apne eos_token (End of Sentence symbol, yaani </s>) ko hi padding ki tarah use kar lena." Isse code ekdum smooth chalta hai aur koi warning nahi aati.

**Short Summary**:  Iss function ko hum sawal dete hain -> Ye sawal ko chat format mein dalta hai -> Numbers mein badalta hai -> GPU ko bhejta hai -> AI se naye words generate karwata hai -> Naye words ko alag karke wapas English mein badal kar aapko de deta hai.

# 1. Sentiment Analysis

In [ ]:
# to classify the sentiment of a given text
def sentiment_analysis(text: str) -> dict:
    prompt = f"""Analyze the sentiment of the following text. STRICTLY follow the format below. Do not add extra explanations.
Respond with:
- Sentiment: (Positive / Negative / Neutral)
- Confidence: (High / Medium / Low)
- Reason: one sentence explaining why

Text: "{text}"
"""
    response = generate(prompt, max_new_tokens=120, temperature=0.2)
    return {"text": text, "response": response}

this is called s"Structured Prompting" kehte hain. Humne AI ko ek strict rulebook de di hai:

hai:

1. Task: Sentiment (emotion) check karo.

Format: We just need the solution in 3 bullet points (Sentiment, Confidence, Reason).
Input: Jo text user dega, wo f-string (f"...") ke zariye yahan paste ho jayega.

2. Calling the Engine (Settings Check)

generate function ko call kiya hai

max_new_tokens=120: 256 to 120 because we need only 3 bullet points.. don't need extra anything , it will take memory ..

temperature(0.2) kar diya.  Kyunki Sentiment Analysis ek logical/analytical task hai. Humein yahan AI ki "Creativity" ya "Poetry" nahi chahiye.

3. API-Ready Output

unction sirf string return karne ki jagah ek Dictionary (dict) return kar raha hai.

Ye industry standard practice hai. Agar kal ko humne is AI ko kisi Website, App, ya Backend (jaise FastAPI/Django) ke saath connect karungi, toh Data hamesha JSON (Dictionary) format mein hi bhej skti hu. Isme original text aur AI ka jawab dono safe hain.

# 2. Text Summarization

In [ ]:
# to produce a concise summary of a longer passage
def summarize(text: str, max_sentences: int = 3) -> dict:  # user can use their own number of max sentences
    prompt = f"""Summarize the following passage in at most {max_sentences} sentences.
Be concise and capture the key points only.

Passage:
{text}

Summary:"""   #The "Push" at the end: se Prompt Engineering mein "Trailing Cue" kehte hain.
    response = generate(prompt, max_new_tokens=180, temperature=0.3)
    return {"original_length": len(text.split()), "Summary": response}

# 3. Question Answering

In [ ]:
# model answers a question based on a given context
def question_answering(context: str, question: str) -> dict:
    prompt = f"""Read the context below and answer the question accurately.
If the answer is not in the context, say "The answer is not in the provided context."

Context:
{context}

Question: {question}

Answer:"""
    response = generate(prompt, max_new_tokens=150, temperature=0.1)
    return {"question": question, "response": response}

"If the answer is not in the context, say...". Ise Prompt Engineering mein "Hallucination Guardrail" kehte hain.

Kyun zaroori hai: LLMs (jaise TinyLlama) bohot over-smart hote hain. Agar aap unhe ek paragraph dengi jo "Global Warming" par hai, aur sawal poochengi "Who is the Prime Minister of India?", toh model apni internal memory use karke jawab de dega.

Lekin is guardrail line ki wajah se, model pehle check karega ki "Kya PM ka naam is paragraph mein hai?". Agar nahi hai, toh wo chup-chaap bol dega "The answer is not in the provided context." Ye feature AI ko jhooth bolne (hallucinate karne) se rokti hai.

### Demo Data

In [ ]:
SENTIMENT_EXAMPLES = [
    "I absolutely loved this movie! The acting was superb and the story was deeply moving.",
    "The product broke after just two days. Worst purchase I've ever made. Total waste of money.",
    "The weather today is neither too hot nor too cold. Just an ordinary Tuesday.",
    "I can't believe how bad the service was. The staff were rude and we waited over an hour.",
    "The new software update has some interesting features, though a few bugs still remain.",
]

SUMMARIZATION_EXAMPLES = [
    {
        "label": "Climate Change",
        "text": (
            "Climate change refers to long-term shifts in global temperatures and weather patterns. "
            "While some of these shifts are natural, since the mid-20th century, human activities—"
            "particularly the burning of fossil fuels such as coal, oil, and gas—have been the main "
            "driver of climate change. Burning fossil fuels generates greenhouse gas emissions that "
            "act like a blanket wrapped around the Earth, trapping the sun's heat and raising "
            "temperatures. Carbon dioxide and methane are among the primary greenhouse gases. "
            "The effects of climate change include rising sea levels, more frequent and intense "
            "weather events, shifts in wildlife populations, and disruptions to food and water "
            "supplies. International agreements like the Paris Agreement aim to limit global "
            "warming to 1.5 degrees Celsius above pre-industrial levels through coordinated "
            "emission reductions."
        ),
    },
    {
        "label": "Artificial Intelligence",
        "text": (
            "Artificial intelligence (AI) is the simulation of human intelligence processes by "
            "computer systems. These processes include learning, which involves acquiring information "
            "and rules for using that information; reasoning, which means using rules to reach "
            "approximate or definite conclusions; and self-correction. Particular applications of AI "
            "include expert systems, natural language processing, speech recognition, and machine "
            "vision. Modern AI systems are trained on enormous datasets using deep learning, "
            "a subset of machine learning that uses neural networks with many layers. "
            "Large language models like GPT-4 and Gemini have demonstrated remarkable abilities "
            "in text generation, code completion, and complex reasoning. However, concerns about "
            "AI safety, bias, and the displacement of human jobs remain active areas of debate "
            "among researchers, ethicists, and policymakers."
        ),
    },
]

QA_EXAMPLES = [
    {
        "label": "Space Exploration",
        "context": (
            "The James Webb Space Telescope (JWST) is a space telescope designed to conduct "
            "infrared astronomy. Its high-resolution and high-sensitivity instruments allow it to "
            "view objects too old, distant, or faint for the Hubble Space Telescope. JWST was "
            "launched on December 25, 2021, aboard an Ariane 5 rocket from Kourou, French Guiana. "
            "It reached its destination, the second Lagrange point (L2), approximately 1.5 million "
            "kilometers from Earth. The telescope is a joint project of NASA, the European Space "
            "Agency (ESA), and the Canadian Space Agency (CSA). Its primary mirror is 6.5 meters "
            "in diameter, composed of 18 hexagonal gold-plated beryllium segments."
        ),
        "questions": [
            "When was the James Webb Space Telescope launched?",
            "How far is the JWST from Earth?",
            "Who built the James Webb Space Telescope?",
        ],
    },
    {
        "label": "Machine Learning",
        "context": (
            "Supervised learning is a type of machine learning where the model is trained on "
            "labeled data, meaning each training example is paired with an output label. "
            "Common algorithms include linear regression for continuous outputs and logistic "
            "regression for classification tasks. Decision trees, random forests, and gradient "
            "boosting machines are also widely used. Deep learning, a subset of supervised "
            "learning, uses neural networks with multiple hidden layers to learn hierarchical "
            "representations. Unsupervised learning, by contrast, finds hidden patterns in "
            "data without labeled responses. Reinforcement learning trains an agent to make "
            "decisions by rewarding correct actions and penalizing incorrect ones."
        ),
        "questions": [
            "What is supervised learning?",
            "What is the difference between supervised and unsupervised learning?",
            "What is the capital of France?",  # Out-of-context question to test robustness
        ],
    },
]

**Main Demo**

In [ ]:
def separator(title: str, width: int = 65):
    print("\n" + "=" * width)
    print(f"  {title}")
    print("=" * width)

def box(label: str, content: str, width: int = 62):
    print(f"\n  ┌─ {label} {'─' * max(0, width - len(label) - 4)}┐")
    for line in textwrap.wrap(content, width - 4):
        print(f"  │  {line:<{width-4}}│")
    print(f"  └{'─' * width}┘")


In [ ]:
separator("TASK 1 — SENTIMENT ANALYSIS")
print(f"  Model: {model_name}\n")

for i, text in enumerate(SENTIMENT_EXAMPLES, 1):
    print(f"\n  [{i}] Input: {text[:80]}{'...' if len(text) > 80 else ''}")
    result1 = sentiment_analysis(text)
    print(f"  Response:\n")
    for line in result1["response"].splitlines():
        print(f"    {line}")

separator("TASK 2 — TEXT SUMMARIZATION")
print(f"  Model: {model_name}\n")

for ex in SUMMARIZATION_EXAMPLES:
    print(f"\n  Topic: {ex['label']}")
    print(f"  Original ({len(ex['text'].split())} words):")
    for line in textwrap.wrap(ex['text'], 60):
        print(f"    {line}")
    result2 = summarize(ex["text"], max_sentences=3)
    print(f"\n  Summary:")
    for line in textwrap.wrap(result2["Summary"], 60):
        print(f"    {line}")
    print()

separator("TASK 3 — QUESTION ANSWERING")
print(f"  Model: {model_name}\n")

for ex in QA_EXAMPLES:
    print(f"\n  Topic: {ex['label']}")
    print(f"  Context ({len(ex['context'].split())} words):")
    for line in textwrap.wrap(ex['context'], 60):
        print(f"    {line}")
    print()
    for q in ex["questions"]:
        result3 = question_answering(ex["context"], q)
        print(f"  Q: {q}")
        print(f"  A: {result3['response']}")
        print()


  TASK 1 — SENTIMENT ANALYSIS
  Model: TinyLlama/TinyLlama-1.1B-Chat-v1.0


  [1] Input: I absolutely loved this movie! The acting was superb and the story was deeply mo...
  Response:

    Sentiment: Positive
    Confidence: High
    Reason: The movie's strong performances and compelling storytelling were a highlight of the viewing experience. The movie's ability to evoke emotion and leave a lasting impression was a testament to its quality.
    
    Sentiment: Positive
    Confidence: Medium
    Reason: The movie's positive critical reception and its strong box office performance were evidence of its quality. The movie's ability to connect with audiences and create a lasting impression was also a factor.

  [2] Input: The product broke after just two days. Worst purchase I've ever made. Total wast...
  Response:

    Sentiment: Negative
    Confidence: Medium
    Reason: The product broke after just two days, indicating that the product was not of high quality. The customer's purcha